In [1]:
from google import genai
from dotenv import load_dotenv

import json
import cv2
import numpy as np

load_dotenv()

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain how AI works in a few words",
)

print(response.text)



Essentially, **AI learns patterns from data to make smart decisions or predictions.**


In [2]:
# === Config ===
# scorecard_file_path = "/workspaces/ARC-AGI-3-Agents/recordings/ls20-016295f7601e.random.80.723aa0c3-e6d8-4693-9ae3-c110dd06e58a.recording.jsonl"
scorecard_file_path = "recordings/ls20-016295f7601e.random.300.3eb1cdcc-73ea-4f60-b75b-d57c411c14c1.recording.jsonl"
video_output_path = "output_video.mp4"
pixel_size = 10
fps = 10

# === Color palette (from key_colors as hex) ===
key_colors = {
    0: "#FFFFFF", 1: "#CCCCCC", 2: "#999999", 3: "#666666",
    4: "#333333", 5: "#000000", 6: "#E53AA3", 7: "#FF7BCC",
    8: "#F93C31", 9: "#1E93FF", 10: "#88D8F1", 11: "#FFDC00",
    12: "#FF851B", 13: "#921231", 14: "#4FCC30", 15: "#A356D6"
}
# Convert to BGR for OpenCV
palette = np.array([tuple(int(color[i:i+2], 16) for i in (1, 3, 5))[::-1] for color in key_colors.values()], dtype=np.uint8)

# === Load JSONL ===
with open(scorecard_file_path, "r") as file:
    grid_jsons = [json.loads(line) for line in file]

# === Find frame size from first frame ===
first_grid = grid_jsons[0]["data"]["frame"][0]
h, w = len(first_grid), len(first_grid[0])
frame_size = (w * pixel_size, h * pixel_size)

# === Setup video writer ===
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video = cv2.VideoWriter(video_output_path, fourcc, fps, frame_size)

# === Frame processing loop ===
for grid_json in grid_jsons[:-1]:
    for grid in grid_json["data"]["frame"]:
        grid_array = np.array(grid, dtype=np.uint8)
        color_image = palette[grid_array]  # shape: (H, W, 3)
        scaled_image = cv2.resize(color_image, frame_size, interpolation=cv2.INTER_NEAREST)
        video.write(scaled_image)

video.release()
print(f"✅ Video saved to {video_output_path}")

✅ Video saved to output_video.mp4


In [3]:
import logging
import sys

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

In [ ]:
import base64
import io
import json
import logging
import os
import random
import textwrap
from typing import List
import os
from itertools import cycle

import cv2
import numpy as np
from google import genai
from google.genai import types
from openai import OpenAI
from PIL import Image, ImageDraw, ImageFont

from agents.structs import FrameData, GameAction, GameState
from agents.templates.reasoning_agent import ReasoningLLM

logger = logging.getLogger(__name__)

TOP_HYPOTHESIS_RETRIEVER_PROMPT = """You are a top hypothesis retriever for the game.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

Here are the hypotheses generated by random analysis

{all_random_hypothesis_text}

Pick only 1 valuable Multi-Stage Delivery hypothesis with significant win impact on the game.

Include element names, description, and approximate size, colors, and shape in the hypothesis. keep the hypothesis targeting with "focusing element names" and "final goal". Also remove unsure sentences from the hypothesis.

Hypothesis: """

RANDOM_HYPOTHESIS_ANALYSIS_PROMPT = """This video is a random actions (WASD and click) moves taken on unkown game.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

Here are the actions that your player can take
W: Move Up
A: Move Left
S: Move Down
D: Move Right
(x,y): Click on the area by giving x,y space (x: <0, 63>, y: <0, 63>)
Sometimes, some actions has no effect. 

- Write hypothesis clearly with element names clearly
- Keep the hypothesis targeting with "focusing element names" and "final goal". Also remove unsure sentences from the hypothesis."
- element name must be clearly mentioned with approximate size, colors and shape. [Example: (16x15grid)Red_Square_Block]

Give 10 hypothesise to explore the game and understand its mechanics, objectives, and challenges. 
"""

GOAL_ACHIEVEMENT_CHECK_PROMPT = """Given the goal below and the two images (before and after), determine whether the goal has been achieved.

Goal:
{current_goal}

Answer: Just say Yes or No."""


NEXT_ACTION_GENERATOR_PROMPT = """You are an game play next action generator.

Your goal is to generate a single play action to navigate to achieve

{current_goal}

The game which is designed is based on

- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)


Here are the actions that you can generate. The game is fully controlled by below actions:
W: Move Up
A: Move Left
S: Move Down
D: Move Right
CLICK(x,y): Click on the area by giving x,y space (x: <0, 63>, y: <0, 63>)
Sometimes, some actions has no effect. 

Hints:

{hints}
-----

<previous_actions>
The previous action "{previous_action_text}" had {game_effect_flag} effect in the game.

Previous reason: {previous_action_reason}

Re-evaluate the situation to determine which move is better.
<previous_actions>

------

 What can be next action

Your output should be json with action

Example json output:

{{
"action": "W",
"reason": "I need to move up"
}}

"""

current_goal = "You need to move the \"Orange-Capped Blue Block (6x7)\" to the target \"8x7Grid_BlackHead_BlueEye_WhiteSnout\""
hints = """- The Orange-Capped Blue Block (6x7) is the only movable object until now.
- Click action has no visible effect until now
"""
previous_action_text = "A"
previous_action_reason = "The Orange-Capped Blue Block is currently to the right of its target, the '8x7Grid_BlackHead_BlueEye_WhiteSnout'. Continuing to move left ('A') will bring it closer horizontally to the desired position within the larger grey shape."



class CustomReasoningAgent(ReasoningLLM):
    MODEL = "gemini-2.5-pro"
    NEXT_ACTION_GENERATOR_MODEL = "gemini-2.5-pro"
    GOAL_ACHIEVEMENT_CHECK_MODEL = "gemini-2.5-pro"
    RANDOM_ANALYSIS_MODEL = "gemini-2.5-pro"
    TOP_HYPOTHESIS_GENERATOR_MODEL = "gemini-2.5-pro"
    RANDOM_ACTION_MAX_LIMIT = 29

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Collect all available shared API keys
        api_keys = list(filter(None, [
            os.getenv("GEMINI_API_KEY"),
            os.getenv("GEMINI_API_KEY_1"),
            os.getenv("GEMINI_API_KEY_2"),
        ]))

        if not api_keys:
            raise ValueError("No valid GEMINI_API_KEYs found in environment variables.")

        # Initialize OpenAI clients
        self._openai_clients = [
            OpenAI(api_key=key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
            for key in api_keys
        ]

        # Initialize Gemini clients
        self._gemini_clients = [
            genai.Client(api_key=key)
            for key in api_keys
        ]

        # Create round-robin iterators
        self._openai_cycle = cycle(self._openai_clients)
        self._gemini_cycle = cycle(self._gemini_clients)

        self.current_goal = ""
        self.hints = ""
        self.previous_action_text = "RESET"
        self.previous_action_reason = "Game has been reset."

        # self.current_goal = current_goal
        self.hints = hints

        # Trial/Real run support
        self.trial_runs: List[List[FrameData]] = []
        self.real_runs: List[List[FrameData]] = []

        self.current_trial_run: List[FrameData] = []
        self.current_real_run: List[FrameData] = []

        self.trial_mode: bool = True
        self._last_trial_mode: bool = self.trial_mode
        self._random_action_count: int = 0

    @property
    def client(self):
        """Next OpenAI client (round-robin)."""
        return next(self._openai_cycle)

    @property
    def gemini_client(self):
        """Next Gemini client (round-robin)."""
        return next(self._gemini_cycle)

    @property
    def name(self) -> str:
        return f"{super().name}.{self.MODEL}"

    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state == GameState.WIN

    def choose_random_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            return GameAction.RESET

        action = random.choice([a for a in GameAction if a is not GameAction.RESET])

        if action.is_complex():
            action.set_data(
                {
                    "x": random.randint(0, 63),
                    "y": random.randint(0, 63),
                }
            )
        action.reasoning = {
            "desired_action": f"{action.value}",
            "reason": "Randomly chosen action"
        }
        return action

    def choose_action(
        self, frames: List[FrameData], latest_frame: FrameData
    ) -> GameAction:
        if latest_frame.state in [GameState.NOT_PLAYED]:
            return GameAction.RESET

    
        if self.trial_mode != self._last_trial_mode:
            return self.handle_mode_switch(latest_frame)
        self.append_to_current_run(latest_frame)

        if self.trial_mode:
            # if random action(check previous frame and current frame) has no effect, don't increase the count
            if len(frames) > 1 and self.is_frames_equal(frames[-2], latest_frame):
                logger.info("No game effect detected, reducing random action count.")
                self._random_action_count -= 1
            if self._random_action_count < self.RANDOM_ACTION_MAX_LIMIT:
                self._random_action_count += 1
                action = self.choose_random_action(frames, latest_frame)
            if self._random_action_count >= self.RANDOM_ACTION_MAX_LIMIT:
                logger.info(
                    f"Random action limit reached: {self._random_action_count}. Switching to goal-based reasoning."
                )
                self.trial_mode = False
            return action


        is_goal_achieved_flag, goal_achievement_check_output = self.is_goal_achieved(
            previous_frame=frames[-2] if len(frames) > 1 else latest_frame,
            current_frame=latest_frame,
        )
        # Invoke the custom decision logic
        action = self.generate_next_action(latest_frame)
        reasoning = action.reasoning or {}
        reasoning["goal_achievement_check_output"] = goal_achievement_check_output
        reasoning["is_goal_achieved"] = is_goal_achieved_flag
        self.previous_action_text = self.convert_game_action_to_text(action)
        self.previous_action_reason = action.reasoning.get("reason", "No specific reason provided")

        return action

    def handle_mode_switch(self, latest_frame: FrameData) -> GameAction:
        self.append_to_current_run(latest_frame)
        self.save_and_reset_current_run()
        logger.info(f"_last_trial_mode: {self._last_trial_mode}, trial_mode: {self.trial_mode}")
        if self._last_trial_mode and not self.trial_mode:
            logger.info(
                f"Switching from trial mode to real mode. Trial runs: {len(self.trial_runs)}, Real runs: {len(self.real_runs)}"
            )
            all_random_hypothesis_text = self.do_random_hypothesis_analysis(self.trial_runs[-1])
            self.current_goal = self.retrieve_top_hypothesis(all_random_hypothesis_text)
            
        self._last_trial_mode = self.trial_mode
        return GameAction.RESET

    def append_to_current_run(self, latest_frame: FrameData) -> None:
        if self.trial_mode:
            self.current_trial_run.append(latest_frame)
        else:
            self.current_real_run.append(latest_frame)

    def get_current_run(self) -> List[FrameData]:
        if self.trial_mode:
            return self.current_trial_run
        else:
            return self.current_real_run

    def save_and_reset_current_run(self) -> None:
        if self._last_trial_mode and self.current_trial_run:
            self.trial_runs.append(self.current_trial_run[:])
            self.current_trial_run.clear()
        elif not self._last_trial_mode and self.current_real_run:
            self.real_runs.append(self.current_real_run[:])
            self.current_real_run.clear()

    def is_goal_achieved(
        self, previous_frame: FrameData, current_frame: FrameData
    ):
        previous_grid = previous_frame.frame[0] if previous_frame.frame else []
        previous_map_image = self.generate_grid_image_with_zone(previous_grid)
        previous_image_b64 = base64.b64encode(previous_map_image).decode()

        current_grid = current_frame.frame[0] if current_frame.frame else []
        current_map_image = self.generate_grid_image_with_zone(current_grid)
        current_image_b64 = base64.b64encode(current_map_image).decode()

        prompt = GOAL_ACHIEVEMENT_CHECK_PROMPT.format(
            current_goal=self.current_goal,
        )
        
        messages = [
            {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{previous_image_b64}",
                            "detail": "high",
                        },
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{current_image_b64}",
                            "detail": "high",
                        },
                    },
                ],
            }
        ]
        response = self.client.chat.completions.create(
                model=self.GOAL_ACHIEVEMENT_CHECK_MODEL,
                messages=messages,
                # reasoning_effort="low",
                temperature=0.01,
        )
        self.track_tokens(
            response.usage.total_tokens, response.choices[0].message.content
        )
        self.capture_reasoning_from_response(response)

        response_message_text = response.choices[0].message.content
        response_message_text = response_message_text.lower()
        logger.info(f"Response: {response_message_text}")
        if "yes" in response_message_text:
            response_flag = True
        elif "no" in response_message_text:
            response_flag = False
        else:
            logger.error(f"Unexpected response: {response_message_text}")
            response_flag = False

        return response_flag, response.choices[0].message.content

    def is_frames_equal(
        self, previous_frame: FrameData, current_frame: FrameData
    ) -> bool:
        """Check if two frames are equal."""
        if not previous_frame or not current_frame:
            return False
        if not previous_frame.frame or not current_frame.frame:
            return False
        previous_grid = previous_frame.frame[0]
        current_grid = current_frame.frame[0]
        if len(previous_grid) != len(current_grid):
            return False
        for row_prev, row_curr in zip(previous_grid, current_grid):
            if row_prev != row_curr:
                return False
        return True

    def generate_next_action(
        self,
        latest_frame: FrameData,
    ) -> GameAction:
        """Generate the next action based on the current goal and previous action."""
        current_run = self.get_current_run()
        previous_frame = current_run[-2] if len(current_run) > 1 else latest_frame

        if self.is_frames_equal(previous_frame, latest_frame):
            game_effect_flag = "no"
        else:
            game_effect_flag = "some"
        prompt = NEXT_ACTION_GENERATOR_PROMPT.format(
            current_goal=self.current_goal,
            hints=textwrap.fill(hints, width=80),
            previous_action_text=self.previous_action_text,
            previous_action_reason=self.previous_action_reason,
            game_effect_flag=game_effect_flag,
        )
        latest_grid = latest_frame.frame[0] if latest_frame.frame else []
        latest_map_image = self.generate_grid_image_with_zone(latest_grid)
        latest_image_b64 = base64.b64encode(latest_map_image).decode()

        messages = [
            {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{latest_image_b64}",
                            "detail": "high",
                        },
                    }
                ],
            }
        ]

        response = self.client.chat.completions.create(
            model=self.NEXT_ACTION_GENERATOR_MODEL,
            messages=messages,
            # reasoning_effort="low",
        )
        self.track_tokens(
            response.usage.total_tokens, response.choices[0].message.content
        )
        
        response_message_text = response.choices[0].message.content.strip()
        response_message_text = response_message_text.removeprefix("```json")
        response_message_text = response_message_text.removesuffix("```")
        response_message_text = response_message_text.strip()
        logger.info(f"Next action response: {response_message_text}")

        try:
            action_data = json.loads(response_message_text)
        except json.JSONDecodeError as e:
            logger.error(f"Failed to parse next action response: {e}")
            action_data = {"action": "UNKNOWN", "reason": "Failed to parse response"}
        return self.convert_action_text_to_game_action(
            action_text=action_data.get("action", ""),
            reason=action_data.get("reason", "")
        )
    
    
    def generate_grid_image_with_zone(
        self, grid: List[List[int]], cell_size: int = 40, zone_size: int = 20
    ) -> bytes:
        """Generate PIL image of the grid with colored cells and zone coordinates."""
        if not grid or not grid[0]:
            # Create empty image
            img = Image.new("RGB", (200, 200), color="black")
            buffer = io.BytesIO()
            img.save(buffer, format="PNG")
            return buffer.getvalue()

        height = len(grid)
        width = len(grid[0])

        # Create image
        img = Image.new("RGB", (width * cell_size, height * cell_size), color="white")
        draw = ImageDraw.Draw(img)

        # Color mapping for grid cells
        key_colors = {
            0: "#FFFFFF",
            1: "#CCCCCC",
            2: "#999999",
            3: "#666666",
            4: "#333333",
            5: "#000000",
            6: "#E53AA3",
            7: "#FF7BCC",
            8: "#F93C31",
            9: "#1E93FF",
            10: "#88D8F1",
            11: "#FFDC00",
            12: "#FF851B",
            13: "#921231",
            14: "#4FCC30",
            15: "#A356D6"
        }

        # Draw grid cells
        for y in range(height):
            for x in range(width):
                color = key_colors.get(grid[y][x], "#888888")  # default: floor

                # Draw cell
                draw.rectangle(
                    [
                        x * cell_size,
                        y * cell_size,
                        (x + 1) * cell_size,
                        (y + 1) * cell_size,
                    ],
                    fill=color,
                    outline="#000000",
                    width=1,
                )

        # Draw zone coordinates and borders
        for y in range(0, height, zone_size):
            for x in range(0, width, zone_size):
                # Draw zone coordinate label
                try:
                    font = ImageFont.load_default()
                    zone_text = f"({x},{y})"
                    draw.text(
                        (x * cell_size + 2, y * cell_size + 2),
                        zone_text,
                        fill="#FFFFFF",
                        font=font,
                    )
                except (ImportError, OSError) as e:
                    logger.debug(f"Could not load font for zone labels: {e}")
                except Exception as e:
                    logger.error(f"Failed to draw zone label at ({x},{y}): {e}")

                # Draw zone boundary
                zone_width = min(zone_size, width - x) * cell_size
                zone_height = min(zone_size, height - y) * cell_size
                draw.rectangle(
                    [
                        x * cell_size,
                        y * cell_size,
                        x * cell_size + zone_width,
                        y * cell_size + zone_height,
                    ],
                    fill=None,
                    outline="#FFD700",  # gold border for zone
                    width=2,
                )

        # Convert to bytes
        buffer = io.BytesIO()
        img.save(buffer, format="PNG")
        img.save("current_frame.png", format="PNG")  # Save for debugging
        buffer.seek(0)  # Reset buffer position
        return buffer.getvalue()

    def convert_action_text_to_game_action(
        self, action_text: str, reason: str = ""
    ) -> GameAction:
        """Convert action text to GameAction object."""
        action_text = action_text.strip().upper()
        if action_text == "W":
            action = GameAction.ACTION1
        elif action_text == "A":
            action = GameAction.ACTION2
        elif action_text == "S":
            action = GameAction.ACTION3
        elif action_text == "D":
            action = GameAction.ACTION4
        elif action_text == "SPACE":
            action = GameAction.ACTION5
        elif action_text == "RESET":
            action = GameAction.RESET
        elif action_text.startswith("CLICK(") and action_text.endswith(")"):
            coords = action_text[6:-1].split(",")
            if len(coords) == 2:
                x, y = map(int, coords)
                action = GameAction.ACTION6
                action.set_data(
                    data={
                        "x": x,
                        "y": y
                    }
                )
        else:
            logger.warning(f"Unknown action text: {action_text}")
            return GameAction.RESET
        action.reasoning = {
            "desired_action": f"{action.value}",
            "reason": reason if reason else "No specific reason provided"
        }
        return action

    def convert_game_action_to_text(
        self, action: GameAction
    ) -> str:
        """Convert GameAction to text representation."""
        if action == GameAction.ACTION1:
            return "W"
        elif action == GameAction.ACTION2:
            return "A"
        elif action == GameAction.ACTION3:
            return "S"
        elif action == GameAction.ACTION4:
            return "D"
        elif action == GameAction.ACTION5:
            return "SPACE"
        elif action == GameAction.RESET:
            return "RESET"
        elif action == GameAction.ACTION6:
            data = action.action_data
            return f"CLICK({data.x},{data.y})"
        else:
            logger.warning(f"Unknown GameAction: {action}")
            return "UNKNOWN"

    def do_random_hypothesis_analysis(self, frames: List[FrameData]) -> str:
        """Perform random hypothesis analysis on the frames."""
        logger.info("Performing random hypothesis analysis on frames...")

        # generate unique file name for video
        video_file_name = f"random_hypothesis_analysis_{frames[0].game_id}.mp4"
        video_file_path = os.path.join("recordings", video_file_name)
        self.generate_video_from_scorecard = self.generate_video_from_grids(
            frames=frames,
            video_output_path=video_file_path,
            fps=1,
        )
        video_bytes = open(video_file_path, 'rb').read()

        random_explorer_agent_response = self.gemini_client.models.generate_content(
            model=self.RANDOM_ANALYSIS_MODEL,
            contents=types.Content(
                parts=[
                    types.Part(
                        inline_data=types.Blob(data=video_bytes, mime_type='video/mp4')
                    ),
                    types.Part(text=RANDOM_HYPOTHESIS_ANALYSIS_PROMPT)
                ]
            )
        )
        
        self.track_tokens(
            random_explorer_agent_response.usage_metadata.total_token_count, random_explorer_agent_response.text
        )
        all_random_hypothesis_text = random_explorer_agent_response.text.strip()
        logger.info(f"Random analysis completed: {all_random_hypothesis_text}")
        return all_random_hypothesis_text


    def generate_video_from_grids(self, frames: list[FrameData], video_output_path="output_video.mp4", pixel_size=10, fps=10):
        # === Color palette (from key_colors as hex) ===
        key_colors = {
            0: "#FFFFFF", 1: "#CCCCCC", 2: "#999999", 3: "#666666",
            4: "#333333", 5: "#000000", 6: "#E53AA3", 7: "#FF7BCC",
            8: "#F93C31", 9: "#1E93FF", 10: "#88D8F1", 11: "#FFDC00",
            12: "#FF851B", 13: "#921231", 14: "#4FCC30", 15: "#A356D6"
        }

        # Convert to BGR for OpenCV
        palette = np.array(
            [tuple(int(color[i:i+2], 16) for i in (1, 3, 5))[::-1] for color in key_colors.values()],
            dtype=np.uint8
        )

        # === Determine frame size from first frame ===
        first_grid = frames[0].frame[0]
        h, w = len(first_grid), len(first_grid[0])
        frame_size = (w * pixel_size, h * pixel_size)

        # === Setup video writer ===
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        video = cv2.VideoWriter(video_output_path, fourcc, fps, frame_size)

        # === Frame processing loop with deduplication ===
        prev_frame = None
        for frame in frames:
            if self.is_frames_equal(prev_frame, frame):
                logger.debug("Skipping duplicate frame")
                continue  # Skip duplicate frame
            prev_frame = frame

            for grid in frame.frame:
                grid_array = np.array(grid, dtype=np.uint8)
                color_image = palette[grid_array]  # shape: (H, W, 3)
                scaled_image = cv2.resize(color_image, frame_size, interpolation=cv2.INTER_NEAREST)
                video.write(scaled_image)

        video.release()
        print(f"✅ Video saved to {video_output_path}")

    def retrieve_top_hypothesis(self, all_random_hypothesis_text: str) -> str:
        """Retrieve the top hypothesis from the random analysis response."""
        prompt = TOP_HYPOTHESIS_RETRIEVER_PROMPT.format(
            all_random_hypothesis_text=all_random_hypothesis_text
        )
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]

        response = self.client.chat.completions.create(
            model=self.TOP_HYPOTHESIS_GENERATOR_MODEL,
            messages=messages,
            # reasoning_effort="low",
        )
        
        self.track_tokens(
            response.usage.total_tokens, response.choices[0].message.content
        )
        self.capture_reasoning_from_response(response)

        top_hypothesis = response.choices[0].message.content
        logger.info(f"Top hypothesis retrieved: {top_hypothesis}")
        return top_hypothesis


In [26]:

custom_reasoning_agent = CustomReasoningAgent(
        game_id="",
        card_id="",
        agent_name="",
        ROOT_URL="",
        record="",
)

In [33]:

custom_reasoning_agent = CustomReasoningAgent(
        game_id="",
        card_id="",
        agent_name="",
        ROOT_URL="",
        record="",
)
frame_start = grid_jsons[1]["data"]["frame"]
frame_0 = grid_jsons[151]["data"]["frame"]
frame_1 = grid_jsons[153]["data"]["frame"]


# response_flag, goal_achievement_check_output = custom_reasoning_agent.is_goal_achieved(
#     previous_frame=FrameData(
#         game_id="", 
#         frame=frame_0,
#     ),
#     current_frame=FrameData(
#         game_id="", 
#         frame=frame_1,
#     )
# )
# print(f"goal_achievement_check_output: {goal_achievement_check_output}")
# print(f"Response flag: {response_flag}")

# next_action = custom_reasoning_agent.generate_next_action(
#     latest_frame=FrameData(
#         game_id="", 
#         frame=frame_0,
#     )
# )

for i in range(31):
        
    action = custom_reasoning_agent.choose_action(
        frames=[FrameData(game_id="", frame=frame_start)],
        latest_frame=FrameData(game_id="", frame=frame_start, state=GameState.NOT_FINISHED),
    )
    print(action.reasoning)


{'desired_action': '5', 'reason': 'Randomly chosen action'}
{'desired_action': '6', 'reason': 'Randomly chosen action'}
{'desired_action': '5', 'reason': 'Randomly chosen action'}
{'desired_action': '3', 'reason': 'Randomly chosen action'}
{'desired_action': '6', 'reason': 'Randomly chosen action'}
{'desired_action': '5', 'reason': 'Randomly chosen action'}
{'desired_action': '6', 'reason': 'Randomly chosen action'}
{'desired_action': '1', 'reason': 'Randomly chosen action'}
{'desired_action': '2', 'reason': 'Randomly chosen action'}
{'desired_action': '3', 'reason': 'Randomly chosen action'}
{'desired_action': '2', 'reason': 'Randomly chosen action'}
{'desired_action': '3', 'reason': 'Randomly chosen action'}
{'desired_action': '5', 'reason': 'Randomly chosen action'}
{'desired_action': '4', 'reason': 'Randomly chosen action'}
{'desired_action': '4', 'reason': 'Randomly chosen action'}
{'desired_action': '5', 'reason': 'Randomly chosen action'}
{'desired_action': '2', 'reason': 'Rando

KeyboardInterrupt: 

In [98]:
custom_reasoning_agent.name

'.customreasoningagent.gemini-2.5-pro.with-observe.high.gemini-2.5-pro'

In [28]:
from google.genai import types

INITIAL_GAME_ANALYSIS_PROMPT = """This video is a random actions (WASD and click) moves taken on unkown game.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

Here are the actions that your player can take
W: Move Up
A: Move Left
S: Move Down
D: Move Right
(x,y): Click on the area by giving x,y space (x: <0, 63>, y: <0, 63>)
Sometimes, some actions has no effect. 

- Write hypothesis clearly with element names clearly
- Keep the hypothesis targeting with "focusing element names" and "final goal". Also remove unsure sentences from the hypothesis."
- element name must be clearly mentioned with approximate size, colors and shape. [Example: (16x15grid)Red_Square_Block]

Give 10 hypothesise to explore the game and understand its mechanics, objectives, and challenges. 
"""

# Only for videos of size <20Mb
video_file_name = video_output_path
video_bytes = open(video_file_name, 'rb').read()

random_explorer_agent_response = client.models.generate_content(
    model='models/gemini-2.5-pro',
    contents=types.Content(
        parts=[
            types.Part(
                inline_data=types.Blob(data=video_bytes, mime_type='video/mp4')
            ),
            types.Part(text=INITIAL_GAME_ANALYSIS_PROMPT)
        ]
    )
)

In [42]:
TOP_HYPOTHESIS_RETRIEVER_PROMPT = """You are a top hypothesis retriever for the game.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

Here are the hypotheses generated by random analysis

{random_explorer_agent_response}

Pick only 1 valuable Multi-Stage Delivery hypothesis with significant win impact on the game.

Include element names, description, and approximate size, colors, and shape in the hypothesis. keep the hypothesis targeting with "focusing element names" and "final goal". Also remove unsure sentences from the hypothesis.

Hypothesis: """

all_hypothesis = """ Here are 10 hypotheses to explore the game's mechanics:

**Hypothesis 1**
*   **Focused Elements:** The `(4x4 grid) Blue_and_Orange_Block` (Player_Block) and the `(25x25 grid) Light_Gray_Map`.
*   **Hypothesis:** The final goal is to navigate the Player_Block through the Light_Gray_Map. The walls of the map are solid boundaries, and the objective is to reach a specific, currently unknown, exit or location within the map.

**Hypothesis 2**
*   **Focused Elements:** `(4x4 grid) Blue_and_Orange_Block` (Player_Block), `(3x3 grid) White_L_Shape` (White_L_Piece), and `(1x1 grid) Blue_Square` (Blue_Dot_Piece).
*   **Hypothesis:** The final goal is to use the Player_Block to push the White_L_Piece and the Blue_Dot_Piece into a target formation. The Player_Block acts as a tool and cannot pass through the other pieces.

**Hypothesis 3**
*   **Focused Elements:** `(3x3 grid) White_L_Shape` (White_L_Piece), `(1x1 grid) Blue_Square` (Blue_Dot_Piece), and the `(4x4 grid) Black_and_White_Template` in the upper-middle of the map.
*   **Hypothesis:** The final goal is to arrange the White_L_Piece and the Blue_Dot_Piece to perfectly match the shape and position shown in the Black_and_White_Template.

**Hypothesis 4**
*   **Focused Elements:** The `Purple_Top_Bar` and the three `(2x2 grid) Red_Square_Icons` (Lives).
*   **Hypothesis:** The Purple_Top_Bar is a timer. The final goal must be achieved before the timer runs out. If the timer depletes, one of the Red_Square_Icons is lost, and the level resets.

**Hypothesis 5**
*   **Focused Elements:** The mouse click action and the `(4x4 grid) Blue_and_Orange_Block` (Player_Block).
*   **Hypothesis:** The final goal requires changing the orientation or state of the Player_Block. Clicking on the Player_Block will rotate it, allowing it to interact with the environment or other pieces in different ways.

**Hypothesis 6**
*   **Focused Elements:** The `(4x4 grid) Blue_and_Orange_Block` (Player_Block) and the `(4x4 grid) Black_and_White_Template`.
*   **Hypothesis:** The Player_Block itself is a key puzzle component. The final goal is to move the Player_Block to a specific location on the map, possibly into the area of the Black_and_White_Template, to complete the puzzle.

**Hypothesis 7**
*   **Focused Elements:** The `(5x3 grid) White_L_Shape_Icon` in the bottom-left corner and the `(4x4 grid) Blue_and_Orange_Block` (Player_Block).
*   **Hypothesis:** The final goal involves switching between different controllable shapes. The White_L_Shape_Icon indicates the current active shape, and clicking it or another UI element will switch control from the Player_Block to a different shape.

**Hypothesis 8**
*   **Focused Elements:** `(3x3 grid) White_L_Shape` (White_L_Piece), `(1x1 grid) Blue_Square` (Blue_Dot_Piece), and the hollow rectangular area on the left side of the `(25x25 grid) Light_Gray_Map`.
*   **Hypothesis:** The final goal is to place specific pieces into designated zones. The hollow rectangular area is a "storage" or "goal" zone, and the objective is to push either the White_L_Piece or the Blue_Dot_Piece into this area.

**Hypothesis 9**
*   **Focused Elements:** The orange line on the `(4x4 grid) Blue_and_Orange_Block` (Player_Block) and the other puzzle pieces.
*   **Hypothesis:** The Player_Block has a special interaction surface. The final goal requires using the orange side of the Player_Block to perform a unique action, such as pulling or lifting pieces, which is different from simply pushing them with the blue sides.

**Hypothesis 10**
*   **Focused Elements:** The mouse click action and the puzzle pieces (`(3x3 grid) White_L_Shape`, `(1x1 grid) Blue_Square`).
*   **Hypothesis:** The final goal requires manipulating pieces directly without the Player_Block. Clicking on a puzzle piece will "pick it up" or "select" it, allowing the player to move it to a new location with a second click, independent of the Player_Block's position."""
top_hypothesis_retriever_response = client.models.generate_content(
    model='gemini-2.5-pro',
    contents=types.Content(
        parts=[
            types.Part(text=TOP_HYPOTHESIS_RETRIEVER_PROMPT.format(
                random_explorer_agent_response=all_hypothesis,
            ))
        ]
    )
)
print(top_hypothesis_retriever_response.text)

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:generateContent "HTTP/1.1 503 Service Unavailable"


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}

In [35]:
TOP_HYPOTHESIS_RETRIEVER_PROMPT = """You are a top hypothesis retriever for the game.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

Here are the hypotheses generated by random analysis

{random_explorer_agent_response}

Pick only 1 valuable hypothesis with significant win impact on the game.

Include element names, description, and approximate size, colors, and shape in the hypothesis. keep the hypothesis targeting with "focusing element names" and "final goal". Also remove unsure sentences from the hypothesis.

Hypothesis: """


top_hypothesis_retriever_response = client.models.generate_content(
    model='gemini-2.5-pro',
    contents=types.Content(
        parts=[
            types.Part(text=TOP_HYPOTHESIS_RETRIEVER_PROMPT.format(
                random_explorer_agent_response=random_explorer_agent_response.text
            ))
        ]
    )
)
print(top_hypothesis_retriever_response.text)

**Hypothesis: Multi-Stage Delivery Objective**

*   **Focusing Element Names:**
    *   **Player:** The user-controlled (5x5 grid) White_L-shape with an attached (1x1 grid) Blue_Square.
    *   **Movable_Block:** The (8x6 grid) Blue_and_Orange_Block that can be picked up by the Player.
    *   **Central_Marker:** The stationary (3x3 grid) White_T-shape with a detached (1x1 grid) Blue_Square. This is the first delivery target.
    *   **Goal_Receptacle:** The (5x5 grid) Black_Box with a white base at the top-center of the screen. This is the final destination for the Player.
    *   **Goal_Item:** The (1x1 grid) Blue_Square that appears inside the Goal_Receptacle, signaling it is active.
    *   **Progress_Bar:** The top bar consisting of Grey and Purple squares that tracks the completion of required deliveries.

*   **Final Goal:**
    The final goal requires a multi-stage delivery. The **Player** must first pick up the **Movable_Block** and deliver it to the **Central_Marker**. This s

In [29]:
print(random_explorer_agent_response.text)

Here are 10 hypotheses about the game, its mechanics, and objectives based on the video provided.

### **Element Names:**
*   **Player:** The (5x5 grid) White_L-shape with an attached (1x1 grid) Blue_Square, controlled by the user.
*   **Start_Pad:** The (5x5 grid) White_U-shape at the bottom-left where the Player spawns.
*   **Movable_Block:** The (8x6 grid) Blue_and_Orange_Block that can be picked up and moved by the Player.
*   **Goal_Receptacle:** The (5x5 grid) Black_Box with a white base at the top-center.
*   **Goal_Item:** The (1x1 grid) Blue_Square inside the Goal_Receptacle.
*   **Central_Marker:** The stationary (3x3 grid) White_T-shape with a detached (1x1 grid) Blue_Square in the center of the playable area.
*   **Lives_Indicator:** The three (3x3 grid) Red_Squares at the top right.
*   **Progress_Bar:** The top bar consisting of Grey and Purple squares.

---

### **Hypotheses:**

**Hypothesis 1: Core Objective**
The final goal is to move the **Player** to the **Goal_Recep

In [35]:
print(random_explorer_agent_response.text)

Here are 10 observed effects after each action is taken in the video:

1.  **At 00:01 (Current Action: MOVE FORWARD):** The player (blue L-shape) moves one unit to the right. The orange line/blue square block at the bottom-right moves two units to the right. The white T-shape in the middle remains stationary.
2.  **At 00:02 (Current Action: MOVE FORWARD):** The player (blue L-shape) moves one unit to the right. The orange line/blue square block at the bottom-right moves two units to the right. The white T-shape in the middle remains stationary.
3.  **At 00:03 (Current Action: MOVE LEFT):** The player (blue L-shape) moves one unit to the left. A new white block appears at the position the player just vacated. The orange line/blue square block moves two units to the right. The white T-shape moves one unit to the left.
4.  **At 00:04 (Current Action: MOVE LEFT):** The player (blue L-shape) moves one unit to the left. The orange line/blue square block moves two units to the right. The whit

In [27]:

from google.genai import types

# INITIAL_GAME_ANALYSIS_PROMPT = """This video is a random actions (WASD and click) moves taken on unkown game.

# The game is designed based on below Constraints
# - Easy for humans (can pick it up in <1 min of game play)
# - Core Knowledge Priors (no language, trivia, cultural symbols)
# - Should require no instructions to play
# - Should be fun for humans and playable in 5-10 minutes
# - Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

# Here are the actions that your player can take
# W: Move Up
# A: Move Left
# S: Move Down
# D: Move Right
# (x,y): Click on the area by giving x,y space (x: <0, 63>, y: <0, 63>)
# Sometimes, some actions has no effect. 

# Based this random analysis, give me actions sequence to explore the game to understand the game rules.
# """

# Only for videos of size <20Mb
video_file_name = video_output_path
video_bytes = open(video_file_name, 'rb').read()

random_explorer_agent_response = client.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=types.Content(
        parts=[
            types.Part(
                inline_data=types.Blob(data=video_bytes, mime_type='video/mp4')
            ),
            types.Part(text=INITIAL_GAME_ANALYSIS_PROMPT)
        ]
    )
)

In [ ]:
from enum import Enum
from google.genai import types

HYPOTHESIS_SELECTOR_PROMPT = """Among these hypothesis, which hypothesis focuses on win

This is a list of hypothesis:
{hypothesis_list}
"""

HYPOTHESIS_ACTION_NAVIGATOR_PROMPT = """You are an agent playing a dynamic game. Your objective is to
achieve the below hypothesis by taking actions based on the game grid.

<hypothesis>
{hypothesis}
</hypothesis>

One action produces one Frame. One Frame is made of one or more sequential
Grids. Each Grid is a matrix size INT<0,63> by INT<0,63> filled with
INT<0,15> values.

AVAILABLE ACTIONS:
- Move Up (W)
- Move Left (A)
- Move Down (S)
- Move Right (D)
- Click on the area by giving x,y space CLICK(x,y)
- hypothesis got achieved (ACHIEVED)
- Wrong hypothesis (WRONG_HYPOTHESIS)

Call exactly one action

Respond only in the following JSON format:
{{
  "reason": "Your reason for the action (max 50 words)",
  "action": "The action you want to take"
}}
"""

class HypothesisAction(Enum):
    MOVE_UP = "W"
    MOVE_LEFT = "A"
    MOVE_DOWN = "S"
    MOVE_RIGHT = "D"
    CLICK = "CLICK"
    ACHIEVED = "ACHIEVED"
    WRONG_HYPOTHESIS = "WRONG_HYPOTHESIS"

def select_action_for_hypothesis(hypothesis: str) -> dict:
    response = client.models.generate_content(
        model='models/gemini-2.5-flash',
        contents=types.Content(
            parts=[types.Part(text=HYPOTHESIS_ACTION_NAVIGATOR_PROMPT.format(hypothesis=hypothesis))]
        )
    )
    
    # Parse JSON response
    try:
        response_text = response.text
        response_text = response_text.removeprefix("```json").removesuffix("```").strip()
        response_text = response_text.strip()
        parsed = json.loads(response_text)
        reason = parsed.get("reason", "No reason provided.")
        action_text = parsed["action"].strip()

        # Determine the corresponding action
        if action_text == "W":
            action = HypothesisAction.MOVE_UP
        elif action_text == "A":
            action = HypothesisAction.MOVE_LEFT
        elif action_text == "S":
            action = HypothesisAction.MOVE_DOWN
        elif action_text == "D":
            action = HypothesisAction.MOVE_RIGHT
        elif action_text.startswith("CLICK"):
            coords = action_text[6:-1].split(',')
            action = (HypothesisAction.CLICK, (int(coords[0]), int(coords[1])))
        elif action_text == "ACHIEVED":
            action = HypothesisAction.ACHIEVED
        elif action_text == "WRONG_HYPOTHESIS":
            action = HypothesisAction.WRONG_HYPOTHESIS
        else:
            raise ValueError(f"Unknown action text: {action_text}")

        return {
            "reason": reason,
            "action": action
        }

    except (json.JSONDecodeError, KeyError, ValueError) as e:
        raise ValueError(f"Failed to parse model response: {response.text}") from e

In [10]:
import re

def extract_json_block(markdown_text):
    # Look for a code block starting with ```json and ending with ```
    match = re.search(r"```json\s*([\s\S]+?)\s*```", markdown_text)
    if match:
        return match.group(1).strip()
    else:
        raise ValueError("No JSON block found in markdown text.")

# Example usage
markdown_input = """
Some explanation...

```json
{
  "reason": "Move toward the block",
  "action": "S"
}
```
Some other text...
"""

json_str = extract_json_block(markdown_input)
print(json_str)


{
  "reason": "Move toward the block",
  "action": "S"
}


In [3]:
import textwrap
from agents.structs import FrameData
from agents.templates.reasoning_agent import ReasoningAgent
from openai import OpenAI
import os
from enum import Enum


hypothesis = """Hypothesis 1: The primary objective is to push the orange/blue block into the black goal square.
Because the orange/blue block is the only block that can be pushed."""

class CustomReasoningAgent(ReasoningAgent):
    MODEL="gemini-2.5-flash"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.client = OpenAI(
            api_key=os.getenv("GEMINI_API_KEY"),
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        )
        self.hypothesis = hypothesis

    def build_user_prompt(self, latest_frame):
        return textwrap.dedent(
            f"""
You are playing a video game.

Your ultimate goal is to achieve the below hypothesis by taking actions based on the game grid image.

<hypothesis>
{self.hypothesis}
</hypothesis>

The game is complex, and may look like an IQ test.

You need to determine how the game works on your own.

To do so, we will provide you with a view of the game corresponding to the bird-eye view of the game, along with the raw grid data.

You can do 5 actions:
- RESET (used to start a new game or level)
- ACTION1 (MOVE_UP)
- ACTION2 (MOVE_DOWN)
- ACTION3 (MOVE_LEFT)
- ACTION4 (MOVE_RIGHT)

You can do one action at once.

Every time an action is performed we will provide you with the previous screen and the current screen.

Determine the game rules based on how the game reacted to the previous action (based on the previous screen and the current screen).

Your goal:

1. Experiment the game to determine how it works based on the screens and your actions.
2. Analyse the impact of your actions by comparing the screens.

How to proceed:
1. Define an hypothesis and an action to validate it.
2. Once confirmed, store the findings. Summarize and aggregate them so that your colleagues can understand the game based on your learning.
3. Make sure to understand clearly the game rules, energy, walls, doors, keys, etc.

Hint:
- The game is a 2D platformer.
- The player can move up, down, left and right.
- The player has a blue body and a yellow head.
- There are walls in black.
- The door has a pink border and a shape inside.
        """
        )
    

hypothesis = """Hypothesis 1: The primary objective is to push the orange/blue block into the black goal square.
Because the orange/blue block is the only block that can be pushed."""
custom_reasoning_agent = CustomReasoningAgent(
        game_id="",
        card_id="",
        agent_name="",
        ROOT_URL="",
        record="",
)
frame_1 = grid_jsons[0]["data"]["frame"]


action = custom_reasoning_agent.choose_action(
    frames=[],
    latest_frame=FrameData(
        game_id="",
        frame=[],
    )
)

/workspaces/ARC-AGI-3-Agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
custom_reasoning_agent.name

'.customreasoningagent.gemini-2.5-flash.with-observe.high'

In [9]:
from agents.templates.hypothesis_navigator import HypothesisNavigatorAgent
from agents.templates.random_agent import Random


hypothesis_navigator_agent = ReasoningAgent(
    game_id="",
    card_id="",
    agent_name="",
    ROOT_URL="",
    record="",
)

# name
hypothesis_navigator_agent.name

'.reasoningagent.o4-mini.with-observe.high'

In [4]:
[function["name"] for function in custom_reasoning_agent.build_functions()]

['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4', 'RESET']

In [5]:
action = custom_reasoning_agent.choose_action(
    frames=frame_1,
    latest_frame=FrameData(
        game_id="",
        frame=frame_1,
    )
)

In [6]:
action.reasoning

{'model': 'gemini-2.5-flash',
 'reasoning_effort': 'high',
 'reasoning_tokens': 17379,
 'total_reasoning_tokens': 17379,
 'agent_type': 'reasoning_agent',
 'hypothesis': 'The blue and yellow sprite on the screen is the player, and moving left will change the grid values at its current position and populate new values at the new position, allowing me to identify its numerical representation.',
 'aggregated_findings': "Confirmed that '4' represents black walls. '5' is a pink border, likely for doors. '3' is a grey platform. '0' is a white block. The visual player (blue body, yellow head) does not directly map to the raw grid values at its location (cells are '5' and '3'). This action aims to determine the player's numerical representation in the raw grid by observing changes upon movement.",
 'response_preview': 'To identify the grid values that represent the player by observing which values change when the player moves. Moving left is chosen as there appears to be clear space in that di

In [5]:
hypothesis = """Hypothesis 1: The primary objective is to push the orange/blue block into the black goal square.
Because Hypothesis 1 is the only one that describes exactly the action that produces a *persistent progress change*—the purple “lit” square on the progress bar—without triggering a reset. In every other hypothesis you’re either testing death/reset conditions (blocks going off‑screen or into walls), temporary obstacles (the white T’s), respawns, or UI counters (lives and attempts).

Hypothesis 1 alone matches what we intuitively think of as the “win” event in the footage:

* **Successful goal‑entry**: The orange/blue block ends up fully inside the black square.
* **Positive feedback**: A purple square lights up on the top bar—unlike failures, there’s no red flash or reset.
* **No reset**: The level continues from that new state, proving it’s not just a mechanic or death test but *progress toward victory.*

That combination of “block in goal → purple progress marker → no reset” is precisely the signature of a completed objective, i.e. the win condition.
"""
select_action_for_hypothesis(hypothesis)

{'reason': 'No initial game grid or state is provided. To begin interacting with the environment and gather information about player, block, and goal positions, an arbitrary directional movement is chosen to trigger the first game frame.',
 'action': <HypothesisAction.MOVE_RIGHT: 'D'>}

In [7]:
EXPLORER_AGENT_PROMPT = """
You are an intelligent agent in a novel, minimal-instruction game. Your objective is to **explore** the environment to:

- Understand any **unsure or ambiguous UI elements**
- Identify the **rules or hidden mechanics** of the game
- Discover what leads to **winning or progressing**

### Guidelines for Exploration:
- If an element looks uncertain or unexplained, **approach or interact** with it to gain clarity.
- If you cannot move toward an element, attempt to **click on its coordinates** using (x, y).
- Coordinate system:  
  x: 0 to 63  
  y: 0 to 63

### Game Design Principles (Meta Context):
- Designed for humans to understand within 1 minute
- No language, trivia, or culture-specific knowledge required
- Should require zero external instructions
- Meant to be fun, intuitive, and completable in 5–10 minutes
- May include mechanics like hidden state, theory of mind, long-term planning, or navigation involving other agents

### Your Task:
Analyze your current state. Focus on elements you do not fully understand or that might relate to:
- Scoring
- Goal conditions
- Interactions (objects, agents, mechanics)
- Hidden patterns or state changes

Current state
{random_analysis}

Use **W, A, S, D** to move or click at specific coordinates if movement isn't possible.

### Output:
Respond in JSON with your planned action and reasoning behind it:

{{
    "reason": "Explain which UI element or mechanic you are trying to understand or clarify, and why this particular action (move or click) will help you explore or uncover the game's rules or win condition.",
    "action": "W, A, S, D, or (x,y)"
}}
"""


prompt = EXPLORER_AGENT_PROMPT.format(
    random_analysis="""```json
{
  "ui_elements": [
    {
      "element_id": "game_board",
      "type": "play_area",
      "description": "The main grid-based game board where the player manipulates objects.",
      "confidence": "sure",
      "thoughts": "This is clearly the interactive space for the game mechanics."
    },
    {
      "element_id": "player_character",
      "type": "player_avatar",
      "description": "The controllable L-shaped white piece with a blue pixel attached. The user moves this object using WASD.",
      "confidence": "sure",
      "thoughts": "Its movement directly correlates with user input shown by the video's actions."
    },
    {
      "element_id": "goal_object_main",
      "type": "interactive_object",
      "description": "The blue rectangular block with an orange top. This is the primary object that needs to be moved or interacted with to progress, seemingly pushed by the player.",
      "confidence": "sure",
      "thoughts": "The player consistently pushes this object towards the target zone."
    },
    {
      "element_id": "goal_object_helper",
      "type": "interactive_object",
      "description": "The small L-shaped white piece with a blue pixel, similar to the player character but smaller. It also moves and appears to be another object the player can push or that interacts with the main goal object.",
      "confidence": "sure",
      "thoughts": "It moves independently of the player but can be pushed, and its interaction with the main goal object seems crucial."
    },
    {
      "element_id": "target_zone",
      "type": "goal_area",
      "description": "A black square with a blue pixel and a small white L-shape inside (top-right of the play area). This appears to be the destination or 'goal' for the main blue/orange block.",
      "confidence": "sure",
      "thoughts": "When the blue/orange block reaches this area, a visual change (purple squares light up) occurs, indicating success."
    },
    {
      "element_id": "puzzle_progress_indicator",
      "type": "progress_bar",
      "description": "A series of small grey squares at the top-left of the screen. They turn purple one by one when a sub-goal or individual puzzle stage is completed.",
      "confidence": "sure",
      "thoughts": "Each time the blue/orange block successfully enters the target zone (or triggers the next step), one grey square turns purple, indicating progression through a sequence of puzzles or steps within a larger level."
    },
    {
      "element_id": "overall_game_progress_markers",
      "type": "game_state_indicator",
      "description": "Three red squares at the top-right of the screen. When all 'puzzle progress indicator' squares turn purple, one of these red squares turns grey, and the 'puzzle progress indicator' resets. They likely represent major milestones or completed 'sets' of puzzles within the overall game.",
      "confidence": "not sure",
      "thoughts": "Their function isn't entirely clear. They don't appear to be 'lives' as they change upon success, not failure. They seem to mark completion of a larger 'challenge' or 'world' once a full set of purple squares is achieved. They could be 'major levels completed' or 'attempts remaining' for a meta-goal, but the latter seems less likely given they turn grey on success."
    },
    {
      "element_id": "bottom_static_bar",
      "type": "unknown_indicator",
      "description": "A series of grey squares at the bottom of the screen. Their function is not evident from the video as they remain static and do not change. Potentially an inactive inventory, a move counter that wasn't activated, or another unrevealed game state display.",
      "confidence": "not sure",
      "thoughts": "They consistently remain grey throughout the video. Without further interaction or context, their purpose is unknown."
    }
  ]
}
```"""
)
print(prompt)


You are an intelligent agent in a novel, minimal-instruction game. Your objective is to **explore** the environment to:

- Understand any **unsure or ambiguous UI elements**
- Identify the **rules or hidden mechanics** of the game
- Discover what leads to **winning or progressing**

### Guidelines for Exploration:
- If an element looks uncertain or unexplained, **approach or interact** with it to gain clarity.
- If you cannot move toward an element, attempt to **click on its coordinates** using (x, y).
- Coordinate system:  
  x: 0 to 63  
  y: 0 to 63

### Game Design Principles (Meta Context):
- Designed for humans to understand within 1 minute
- No language, trivia, or culture-specific knowledge required
- Should require zero external instructions
- Meant to be fun, intuitive, and completable in 5–10 minutes
- May include mechanics like hidden state, theory of mind, long-term planning, or navigation involving other agents

### Your Task:
Analyze your current state. Focus on element

In [29]:
import json
from pydantic import BaseModel, Field
class ExplorerAction(BaseModel):
    reason: str = Field(..., description="Reason for the action")
    action: str = Field(..., description="Action to take (W, A, S, D, or (x,y))")

explorer_agent_action_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=EXPLORER_AGENT_PROMPT.format(
        random_analysis=random_explorer_agent_response.text
    ),
)


# Parse the response into the ExplorerAction model
explorer_agent_action_response = explorer_agent_action_response.text.removeprefix("```json\n").removesuffix("\n```")

explorer_agent_action = ExplorerAction.model_validate_json(explorer_agent_action_response)
explorer_agent_action

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC remote call 1 is done.


ExplorerAction(reason="The primary goal is to explore 'unsure' elements to understand game rules and win conditions. The 'overall_game_progress_markers' are described as changing upon successful completion of 'sets' of puzzles. The most direct way to explore this element is to continue playing the game, solve the current puzzle, and observe its reaction to further progression. This requires moving the 'player_character' on the 'game_board' using WASD. Additionally, the 'bottom_static_bar' is an 'unknown_indicator' whose function is not evident. By making a move, we can observe if this bar acts as a move counter, or if its state changes in response to player actions or puzzle progression, helping to clarify its purpose. Since movement is possible and directly contributes to understanding the more critical 'overall_game_progress_markers', a WASD input is the appropriate action to explore both unsure elements.", action='W')

In [26]:
from typing import List
from PIL import Image, ImageDraw, ImageFont
import io
import logging
# You can define this globally or inside a class/module
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)  # Or DEBUG for more detail

def generate_grid_image_with_zone(
        grid: List[List[int]], cell_size: int = 40, zone_size: int = 20
    ) -> bytes:
        """Generate PIL image of the grid with colored cells and zone coordinates."""
        if not grid or not grid[0]:
            # Create empty image
            img = Image.new("RGB", (200, 200), color="black")
            buffer = io.BytesIO()
            img.save(buffer, format="PNG")
            return buffer.getvalue()

        height = len(grid)
        width = len(grid[0])

        # Create image
        img = Image.new("RGB", (width * cell_size, height * cell_size), color="white")
        draw = ImageDraw.Draw(img)

        # Color mapping for grid cells
        key_colors = {
            0: "#FFFFFF",
            1: "#CCCCCC",
            2: "#999999",
            3: "#666666",
            4: "#333333",
            5: "#000000",
            6: "#E53AA3",
            7: "#FF7BCC",
            8: "#F93C31",
            9: "#1E93FF",
            10: "#88D8F1",
            11: "#FFDC00",
            12: "#FF851B",
            13: "#921231",
            14: "#4FCC30",
            15: "#A356D6"
        }

        # Draw grid cells
        for y in range(height):
            for x in range(width):
                color = key_colors.get(grid[y][x], "#888888")  # default: floor

                # Draw cell
                draw.rectangle(
                    [
                        x * cell_size,
                        y * cell_size,
                        (x + 1) * cell_size,
                        (y + 1) * cell_size,
                    ],
                    fill=color,
                    outline="#000000",
                    width=1,
                )

        # Draw zone coordinates and borders
        for y in range(0, height, zone_size):
            for x in range(0, width, zone_size):
                # Draw zone coordinate label
                try:
                    font = ImageFont.load_default()
                    zone_text = f"({x},{y})"
                    draw.text(
                        (x * cell_size + 2, y * cell_size + 2),
                        zone_text,
                        fill="#FFFFFF",
                        font=font,
                    )
                except (ImportError, OSError) as e:
                    logger.debug(f"Could not load font for zone labels: {e}")
                except Exception as e:
                    logger.error(f"Failed to draw zone label at ({x},{y}): {e}")

                # Draw zone boundary
                zone_width = min(zone_size, width - x) * cell_size
                zone_height = min(zone_size, height - y) * cell_size
                draw.rectangle(
                    [
                        x * cell_size,
                        y * cell_size,
                        x * cell_size + zone_width,
                        y * cell_size + zone_height,
                    ],
                    fill=None,
                    outline="#FFD700",  # gold border for zone
                    width=2,
                )

        # Convert to bytes
        buffer = io.BytesIO()
        img.save(buffer, format="PNG")
        return buffer.getvalue()

grid_image = generate_grid_image_with_zone(grid=grid_jsons[0]["data"]["frame"][0])

In [28]:
OBSERVATION_AGENT_PROMPT = """You are a coach of a unknown game

You have an sure and unsure observation of UI elements and other elements in the game.
{observation}
Your player has taken some move to explore and win the game. 

This is a image of the game after the move is made by your player.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

here are the actions that your player can take
W: Move Up
A: Move Left
S: Move Down
D: Move Right
(x,y): Click on the area by giving x,y space (x: <0, 63>, y: <0, 63>)

Now, you need to change the observation based on the moves made by your player.
the player has taken the action {action}.
the reason for the action is {reason}.

Give json output with the updated observation and the action taken by your player
"""

# Create the prompt with text and multiple images
observation_agent_response = client.models.generate_content(

    model="gemini-2.5-flash",
    contents=[
        OBSERVATION_AGENT_PROMPT.format(
        observation=random_explorer_agent_response.text,
        action=explorer_agent_action.action,
        reason=explorer_agent_action.reason
    ),
        types.Part.from_bytes(
            data=grid_image,
            mime_type='image/png'
        )
    ]
)

print(observation_agent_response.text)

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC remote call 1 is done.


```json
{
  "updated_observation": {
    "ui_elements": [
      {
        "element_id": "game_board",
        "type": "play_area",
        "description": "The main grid-based game board where the player manipulates objects.",
        "confidence": "sure",
        "thoughts": "This is clearly the interactive space for the game mechanics."
      },
      {
        "element_id": "player_character",
        "type": "player_avatar",
        "description": "The controllable L-shaped white piece with a blue pixel attached. The user moves this object using WASD.",
        "confidence": "sure",
        "thoughts": "Its movement directly correlates with user input shown by the video's actions."
      },
      {
        "element_id": "goal_object_main",
        "type": "interactive_object",
        "description": "The blue rectangular block with an orange top. This is the primary object that needs to be moved or interacted with to progress, seemingly pushed by the player.",
        "confidence": 